<p style="font-size:16pt; font-weight:bold; color:red; padding-bottom:20px; float:right">Please rename this file before editing!</p>

# Session 9 — A Worked Example: 1.3 Million Consumer Complaints

Notebook 01 introduced the ideas on data you could read. This notebook uses
them on data you cannot: every complaint about a financial product that the
U.S. Consumer Financial Protection Bureau published from December 2011 to May
2019.

**The question:** *Who gets complained about, about what, and is it getting
worse?*

That is three questions, and they are of three different kinds. Naming the
kind decides the summary number and the chart:

| Step | The question | Its kind | Number | Chart |
|---|---|---|---|---|
| 1 | What is in the file? | — | shape, types, missing | — |
| 2 | How many complaints does a company get? | distribution | mean, median, quartiles | histogram |
| 3 | Which companies are far from the rest? | outliers | fences | — |
| 4 | Which products draw the most complaints? | comparison | one count per group | bars |
| 5 | Is it getting worse? | trend | counts per month | line |
| 6 | Do the most-complained-about companies answer late? | relationship | correlation | scatter plot |
| 7 | What goes in the report? | — | — | one figure, four Axes |

Along the way the data sets four traps. Each one produces a chart that looks
fine and says something false.

> **Run this on ARC, with the kernel set to Conda Python 3.12.** The file is
> under `/data/public/` and exists only there. Loading it takes a few seconds.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.options.display.float_format = "{:,.2f}".format
pd.options.display.max_colwidth = 60

PATH = "/data/public/ConsumerComplaintDatabase/rows.csv"

---

## Step 1 — Load and inspect

The file is 737 MB, most of it the text of the complaints. `usecols` reads
only the columns this analysis needs, which is the difference between three
seconds and a minute.

In [ ]:
complaints = pd.read_csv(PATH, usecols=[
    "Date received", "Product", "Company", "State", "Submitted via",
    "Date sent to company", "Timely response?", "Complaint ID"])

complaints.columns = ["received", "product", "company", "state", "via",
                      "sent", "timely", "id"]
complaints["received"] = pd.to_datetime(complaints["received"], format="%m/%d/%Y")
complaints["sent"] = pd.to_datetime(complaints["sent"], format="%m/%d/%Y")

print(complaints.shape)
complaints.head()

One row is one complaint. The Session 8 habits first: types, missing values,
and the range of the dates.

In [ ]:
print(complaints.dtypes)
print()
print(complaints.isna().sum())
print()
print("from", complaints["received"].min().date(), "to", complaints["received"].max().date())
print(complaints["company"].nunique(), "companies,", complaints["product"].nunique(), "products")

1,282,355 complaints against 5,275 companies. Only `state` has missing values.
Note the last date, **10 May 2019**: the final month of the file is not a
whole month. That will matter in Step 5.

---

## Step 2 — A distribution: complaints per company

"How many complaints does a company get?" First reduce the table to one number
per company. `value_counts` is the shortest `groupby` there is.

In [ ]:
per_company = complaints["company"].value_counts()
per_company.head(8)

In [ ]:
print("companies ", len(per_company))
print("mean      ", round(per_company.mean(), 1))
print("median    ", per_company.median())
print("std       ", round(per_company.std(), 1))
print("smallest  ", per_company.min(), "  largest", per_company.max())
print()
print("companies below the mean: {:.0%}".format((per_company < per_company.mean()).mean()))
print("companies with exactly one complaint:", (per_company == 1).sum())

The mean is 243 and the median is 6. "A company gets 243 complaints on
average" is arithmetically true, and 94 % of the companies are below it. A
typical company in this file got six complaints in seven and a half years.
More than a thousand got exactly one.

When mean and median are forty times apart, the data is heavily skewed, and
you know it before drawing anything. The standard deviation, 3,290, is
thirteen times the mean: as a description of a "typical distance from the
centre" it is useless here, because it is built from squared distances and the
few giants supply nearly all of them.

Now the picture. The obvious histogram first.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.hist(per_company, bins=40, color="steelblue", edgecolor="white")
ax.set_xlabel("complaints against the company")
ax.set_ylabel("companies")
ax.set_title("Equal bins")
fig.tight_layout()

**Trap 1: the axis that hides the data.** One bar. Each bin is about 2,900
complaints wide, so nearly every company is in the first one, and the few
giants are spread too thinly along the rest to rise above the axis line. The
chart is correct and shows nothing.

The data spans five powers of ten, from 1 to 115,703. That calls for a
logarithmic axis, with bin edges spaced evenly on it.

In [ ]:
edges = np.geomspace(1, per_company.max(), 25 + 1)

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.hist(per_company, bins=edges, color="steelblue", edgecolor="white")
ax.set_xscale("log")
ax.axvline(per_company.mean(), color="firebrick", linestyle="--",
           label="mean {:,.0f}".format(per_company.mean()))
ax.axvline(per_company.median(), color="black", linestyle=":",
           label="median {:,.0f}".format(per_company.median()))
ax.set_xlabel("complaints against the company (logarithmic scale)")
ax.set_ylabel("companies")
ax.set_title("Most companies draw a handful of complaints; a few draw 100,000")
ax.legend()
fig.tight_layout()

Now every company is visible: the tall bars of companies with one, two, three
complaints at the left, a long slope, and a thin tail reaching 100,000. The
mean sits far out in that tail, to the right of almost every company it is
supposed to describe.

Read a logarithmic axis with care. The distance from 1 to 10 is the same as
from 10,000 to 100,000. The label says so, and it has to.

**✏️ Try it:** Redraw the logarithmic histogram with 8 bins, then with 80. With 80, gaps appear at the left. Why? (How many different values can a count take between 1 and 10?)

---

## Step 3 — Outliers: who is beyond the fences?

In [ ]:
q1, q3 = per_company.quantile(0.25), per_company.quantile(0.75)
iqr = q3 - q1
upper_fence = q3 + 1.5 * iqr
print("q1", q1, " q3", q3, " iqr", iqr, " upper fence", upper_fence)

outside = per_company > upper_fence
print()
print("companies beyond the fence:", outside.sum(), "of", len(per_company))
print("their share of all complaints: {:.0%}".format(
    per_company[outside].sum() / per_company.sum()))

The rule of thumb flags 778 companies, every one with more than 72 complaints.
Together they account for 96 % of the complaints in the file.

So what would "removing the outliers" do here? It would delete Equifax,
Experian, TransUnion, Bank of America and Wells Fargo, and leave a tidy table
describing 4 % of the complaints. The flagged rows are not errors. **They are
the subject.**

That is the usual situation with skewed data, and it is why the rule is used
to *point*, not to *cut*. What it does tell you is that this is two
populations in one column: a few hundred companies that deal with millions of
customers, and thousands of small ones. A single mean, or a single chart on a
linear axis, cannot describe both.

In [ ]:
top = per_company.head(10)
print("the ten largest hold {:.0%} of all complaints".format(top.sum() / per_company.sum()))
top

---

## Step 4 — A comparison: which products?

One count per product, sorted, as bars. The product names are long, so the
bars are horizontal (`ax.barh`), where the labels have room.

In [ ]:
by_product = complaints["product"].value_counts()

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.barh(by_product.index[::-1], by_product.values[::-1], color="steelblue")
ax.set_xlabel("complaints, December 2011 to May 2019")
ax.set_ylabel("product, as named in the file")
ax.set_title("Mortgages draw the most complaints. Or do they?")
fig.tight_layout()

The chart is honest in every way the checklist of notebook 01 can test: bars
from zero, labels, a title. And its message is wrong.

**Trap 2: the categories changed.** Read the labels again. There is "Credit
reporting", and there is "Credit reporting, credit repair services, or other
personal consumer reports". There is "Credit card", "Prepaid card", and
"Credit card or prepaid card". Look at when each name was in use:

In [ ]:
year = complaints["received"].dt.year
names = ["Credit reporting",
         "Credit reporting, credit repair services, or other personal consumer reports",
         "Credit card", "Prepaid card", "Credit card or prepaid card"]
pd.crosstab(year, complaints["product"])[names]

In April 2017 the Bureau reorganised its list of products. The old names stop
in 2017 and the new ones start. One subject was split over two bars, and the
bar chart ranked the halves separately.

No statistic and no chart rule catches this. It is found by reading the
category names, which is one more reason to look at a bar chart's labels and
not only its bars. The repair is a mapping from every name to one group, old
and new alike:

In [ ]:
GROUPS = {
    "Credit reporting": "Credit reporting",
    "Credit reporting, credit repair services, or other personal consumer reports": "Credit reporting",
    "Credit card": "Credit or prepaid card",
    "Prepaid card": "Credit or prepaid card",
    "Credit card or prepaid card": "Credit or prepaid card",
    "Bank account or service": "Bank account",
    "Checking or savings account": "Bank account",
    "Payday loan": "Consumer, payday or vehicle loan",
    "Payday loan, title loan, or personal loan": "Consumer, payday or vehicle loan",
    "Consumer Loan": "Consumer, payday or vehicle loan",
    "Vehicle loan or lease": "Consumer, payday or vehicle loan",
    "Money transfers": "Money transfer or virtual currency",
    "Virtual currency": "Money transfer or virtual currency",
    "Money transfer, virtual currency, or money service": "Money transfer or virtual currency",
}
complaints["group"] = complaints["product"].replace(GROUPS)

by_group = complaints["group"].value_counts()

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(by_group.index[::-1], by_group.values[::-1], color="steelblue")
ax.set_xlabel("complaints, December 2011 to May 2019")
ax.set_ylabel("product group (old and new names combined)")
ax.set_title("Credit reporting, not mortgages, draws the most complaints")
fig.tight_layout()

Credit reporting is first, with 366,000 complaints against 278,000 for
mortgages. The first chart had it third and fourth.

The y label now says what was done to the categories. A reader who compares
this chart with the Bureau's own tables needs to know.

**✏️ Try it:** `complaints['group'].value_counts(normalize=True)` gives shares. Draw the bars as percentages of all complaints. Does the x axis still have to start at zero?

---

## Step 5 — A trend: is it getting worse?

Complaints per month. `dt.to_period("M")` turns each date into its month.

In [ ]:
per_month = complaints["received"].dt.to_period("M").value_counts().sort_index()

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.plot(per_month.index.to_timestamp(), per_month.values, color="steelblue")
ax.set_xlabel("month the complaint was received")
ax.set_ylabel("complaints per month")
ax.set_title("Complaints per month")
fig.tight_layout()

A steady rise for seven years, and then complaints collapse to almost nothing
in the last month. Did American finance fix itself in May 2019?

**Trap 3: the incomplete last period.** Step 1 found that the file ends on 10
May 2019. The last point counts ten days and is drawn beside points that count
thirty. The same thing happens at the start: the first point, December 2011,
is the month the Bureau began publishing.

This is the most common false "trend" in existence. Any chart of counts per
period needs every period to be complete.

In [ ]:
print(per_month.tail(3))

whole_months = per_month[(per_month.index >= "2012-01") & (per_month.index <= "2019-04")]

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.plot(whole_months.index.to_timestamp(), whole_months.values, color="steelblue")
ax.set_ylim(bottom=0)
ax.set_xlabel("month the complaint was received (whole months only)")
ax.set_ylabel("complaints per month")
ax.set_title("Complaints rose from 3,000 a month in 2012 to about 20,000 by 2017")
fig.tight_layout()

A line does not have to start at zero. Here it does anyway (`set_ylim(bottom=
0)`), because the title makes a claim about the *size* of the rise, a factor
of six, and with zero on the axis the eye can check that claim.

Now the spike. One month in 2017 stands well above its neighbours. An outlier
in a time series gets the same treatment as any other: look before deciding.

In [ ]:
print(whole_months.sort_values(ascending=False).head(3))

september = complaints[complaints["received"].dt.to_period("M") == "2017-09"]
print()
print(september["company"].value_counts().head(4))

In [ ]:
per_day = september["received"].dt.day.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.bar(per_day.index, per_day.values, color="steelblue")
ax.set_xlabel("day of September 2017")
ax.set_ylabel("complaints received")
ax.set_title("3,553 complaints on 8 September 2017")
fig.tight_layout()

On 7 September 2017 Equifax announced that the personal data of about 147
million people had been stolen from it. The next day the Bureau received 3,553
complaints, against 250 to 750 on the days before, and Equifax alone drew
9,736 that month.

The spike is real, and it is the most interesting point on the chart. A rule
that "cleans outliers" from a time series would have removed the one month in
which something happened.

Is it getting worse, then? Complaints *received* rose sixfold. Whether
companies behaved six times worse is another matter: over the same years the
Bureau became known, added products, and made filing on the web easy. A count
of complaints measures complaining. The chart is honest about what it counts,
and the caption should not claim more than that.

**✏️ Try it:** Draw one line per product group for the three largest groups (`pd.crosstab(month, complaints['group'])` gives one column per group). Which group does the 2017 spike belong to?

---

## Step 6 — A relationship: do the big ones answer late?

Each complaint records whether the company responded on time. Do companies
with more complaints do worse? One row per company: the number of complaints,
and the share answered on time. The mean of a True/False column is the share
of True.

In [ ]:
complaints["on_time"] = complaints["timely"] == "Yes"
print("answered on time, all complaints: {:.1%}".format(complaints["on_time"].mean()))

companies = complaints.groupby("company").agg(
    complaints=("id", "size"), on_time=("on_time", "mean"))
companies.sort_values("complaints", ascending=False).head()

Before plotting 5,275 points, think about the small companies. A company with
one complaint has an on-time share of either 0 % or 100 %, and nothing in
between is possible. Shares computed from a handful of cases are mostly noise.
(Session 8 met the same problem with restaurants that had three reviews.)

So: only companies with at least 100 complaints.

In [ ]:
enough = companies[companies["complaints"] >= 100]
print(len(enough), "companies with at least 100 complaints")
print(enough["on_time"].describe())

In [ ]:
r = enough["complaints"].corr(enough["on_time"])

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.scatter(enough["complaints"], 100 * enough["on_time"], color="steelblue",
           alpha=0.5, label="r = {:.2f}".format(r))
ax.set_xscale("log")
ax.set_xlabel("complaints against the company (logarithmic scale)")
ax.set_ylabel("answered on time (%)")
ax.set_title("The companies that answer late are small ones, not the largest")
ax.legend()
fig.tight_layout()

r is 0.08: as a straight line, nothing. But look at the picture, because it
says more than "nothing".

- The largest companies, at the right, are all near 100 %. They have whole
  departments for this.
- Most of the others are near 100 % too: the median company answers 98.5 %
  on time.
- Scattered below is a group that answers late most of the time, and all of
  them are at the left, with a few hundred complaints each.

That is a clear pattern, and it is not a line, so r does not see it. This is
Anscombe's point from notebook 01 on real data: **a correlation near zero
means "no straight-line relationship", not "no relationship".**

Who is at the bottom?

In [ ]:
late = enough.sort_values("on_time").head(8)
late.assign(on_time=(100 * late["on_time"]).round(1))

One lender received 369 complaints and answered none of them on time. That row
is an outlier on the y axis, and again it is not noise. It is the row a
regulator would want first.

**✏️ Try it:** Compute r again with the logarithm of the complaint count: `np.log10(enough['complaints']).corr(enough['on_time'])`. It is larger. Does a larger r make the picture any different?

---

## A fourth trap, in passing: impossible values

How long does the Bureau take to pass a complaint on to the company?

In [ ]:
days = (complaints["sent"] - complaints["received"]).dt.days

print(days.describe())
print()
print("passed on the same day: {:.0%}".format((days == 0).mean()))
print("negative:", (days < 0).sum())

Median 0, mean 3.2, maximum 1,962: skewed again, as waiting times nearly
always are.

And 7,052 complaints were sent to the company **the day before they were
received**. That is impossible, so it is an error. What kind?

In [ ]:
early = complaints[days < 0]
print(early["received"].dt.year.value_counts().sort_index())
print()
print(early["via"].value_counts())

**Trap 4: deleting what you have not understood.** All of them are exactly one
day early, all are from 2011 to 2014, and nearly all came in through the web
form. That is not 7,052 separate typing mistakes. It has the fingerprint of a
single cause: two systems that recorded the date in different time zones, for
instance, until someone fixed it in 2014.

What to do depends on the question. For "how long does forwarding take?",
treating them as zero days is defensible; dropping them is too; leaving −1 in
the mean is not. Whatever is done, it gets written down next to the result,
with the count: *7,052 records with a forwarding time of −1 day were counted
as 0.*

---

## Step 7 — The figure for the report

Four findings, four Axes, one Figure. Each chart is drawn by a small function
that takes an `ax`, which is what makes it possible to arrange them freely.

In [ ]:
def draw_companies(ax):
    ax.hist(per_company, bins=np.geomspace(1, per_company.max(), 26),
            color="steelblue", edgecolor="white")
    ax.set_xscale("log")
    ax.axvline(per_company.median(), color="black", linestyle=":",
               label="median {:,.0f}".format(per_company.median()))
    ax.axvline(per_company.mean(), color="firebrick", linestyle="--",
               label="mean {:,.0f}".format(per_company.mean()))
    ax.set_xlabel("complaints against the company (log scale)")
    ax.set_ylabel("companies")
    ax.set_title("A typical company: 6 complaints")
    ax.legend()
    return ax


def draw_groups(ax):
    ax.barh(by_group.index[::-1], by_group.values[::-1] / 1000, color="steelblue")
    ax.set_xlabel("complaints (thousands)")
    ax.set_ylabel("product group")
    ax.set_title("Credit reporting leads")
    return ax


def draw_months(ax):
    ax.plot(whole_months.index.to_timestamp(), whole_months.values / 1000,
            color="steelblue")
    ax.set_ylim(bottom=0)
    ax.annotate("Equifax breach\nSeptember 2017", xy=(pd.Timestamp("2017-09-01"), 27.4),
                xytext=(pd.Timestamp("2013-06-01"), 24),
                arrowprops={"arrowstyle": "->"})
    ax.set_xlabel("month received (whole months only)")
    ax.set_ylabel("complaints per month (thousands)")
    ax.set_title("Sixfold rise, 2012 to 2017")
    return ax


def draw_on_time(ax):
    ax.scatter(enough["complaints"], 100 * enough["on_time"], color="steelblue",
               alpha=0.5)
    ax.set_xscale("log")
    ax.set_xlabel("complaints against the company (log scale)")
    ax.set_ylabel("answered on time (%)")
    ax.set_title("Late answers: the smaller companies")
    return ax


fig, axes = plt.subplots(2, 2, figsize=(13, 8.5))
draw_companies(axes[0, 0])
draw_groups(axes[0, 1])
draw_months(axes[1, 0])
draw_on_time(axes[1, 1])
fig.suptitle("Consumer complaints to the CFPB, December 2011 to May 2019 "
             "(1,282,355 complaints, 5,275 companies)")
fig.tight_layout()

In a program this figure would now be saved and closed:

```python
fig.savefig("complaints_overview.png", dpi=150, bbox_inches="tight")
plt.close(fig)
```

### What an honest statement of the result includes

> From December 2011 to May 2019 the Bureau published 1,282,355 complaints
> against 5,275 companies. Complaints are extremely concentrated: the median
> company received 6, while ten companies received 52 % of the total. Credit
> reporting drew the most complaints (366,000) once the product names used
> before and after April 2017 are combined. Monthly complaints rose about
> sixfold between 2012 and 2017; the peak, September 2017, follows the Equifax
> data breach. 97.5 % of complaints were answered on time; the companies that
> often answered late had a few hundred complaints each, not tens of
> thousands. The count of complaints measures complaints filed, not
> misconduct.

Every sentence of that carries something this notebook had to find out the
hard way: which average, which categories, which months, which companies were
left out and why.

### The four traps

| Trap | What the chart showed | How it was caught |
|---|---|---|
| The axis that hides the data | one bar | mean and median 40 times apart, before drawing |
| Categories that changed | mortgages in first place | reading the labels; a table of name by year |
| The incomplete last period | a collapse in May 2019 | the date range, in Step 1 |
| Deleting what is not understood | (nothing: it would have gone unseen) | asking what the impossible values have in common |

None of the four raised an error or a warning.

---

## Exercises

All on the `complaints` table. For each, decide the kind of question first:
distribution, comparison, relationship or trend.

### 🧩 Exercise 1 — Complaints per state

Count the complaints per `state`. Compute the mean and the median number per
state, and draw the distribution. Then draw the fifteen states with the most
complaints as bars. California is first. What would you have to divide by
before saying that Californians complain more than anyone else?

### 🧩 Exercise 2 — How complaints arrive

`via` says how each complaint was submitted. Draw the share submitted through
the web for each year from 2012 to 2018 (whole years only). Which kind of
chart, and does the y axis have to start at zero?

### 🧩 Exercise 3 — Forwarding times

Take `days` from the fourth trap, with the −1 values set to 0. Compute its
mean, median and the 0.9 and 0.99 quantiles. Draw its distribution in a way
that shows both the 62 % at zero and the tail beyond 100 days. (A logarithmic
axis cannot show zero. One way: draw `days + 1` and say so in the label.)

### 🧩 Exercise 4 — One company over time

Pick one of the ten largest companies and draw its complaints per month, whole
months only. Mark its largest month, find out what is in it (which product
groups?), and put one sentence in the title.

### 🧩 Exercise 5 — Audit your own figure

Take any figure you drew above and go through the checklist at the end of
notebook 01, item by item. Then ask the Axes itself: `ax.get_xlabel()`,
`ax.get_title()`, `ax.get_ylim()`, `ax.get_xscale()`. Which items of the
checklist could a program test, and which need you?

---

**Next:** `03_Statistics_and_Plotting_Reference` lists every method used
here, by task. Homework 8 asks the same four kinds of question of the
invasive-plants data.